In [1]:
# Model은 ChatOpenAI랑 huggingface 쓸 예정

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 캐싱
# 메모리 캐싱
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache

set_llm_cache(InMemoryCache())

In [2]:
# SQLite Cache
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
import os

if not os.path.exists("cache"):
    os.makedirs("cache")

set_llm_cache(SQLiteCache(database_path='cache/tinygpt2_cache.db'))

In [8]:
# 모델 직렬화
# 직렬화가 가능한지 확인
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini',
                 max_tokens=1024)
print(f'chatopenai: {llm.is_lc_serializable()}')

chatopenai: True


In [9]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "{fruit}의 색상이 무엇인가요?"
)

chain = prompt | llm

In [10]:
# 저장
import pickle
import json
from langchain_core.load import dumpd, dumps
from langchain_core.load import loads, load


In [12]:
# 방법1 그냥 불러와서 비동기실행을 한다.
dumps_chain = dumps(chain)
with open("chain.json", 'w') as f:
    f.write(dumps_chain)

with open("chain.json", "r") as f:
    s_chain = f.read()


chain = loads(s_chain, allowed_objects='all')
# chain
await chain.ainvoke({"fruit": "사과"})

AIMessage(content='사과의 색상은 품종에 따라 다양하지만 일반적으로는 빨간색, 녹색, 노란색 등이 있습니다. 예를 들어, 레드 딜리셔스는 주로 붉은색을 띠고, 그라니 스미스는 녹색을 띠며, 골든 딜리셔스는 노란색을 가집니다. 어떤 사과는 이들 색상이 조합된 형태로도 나타납니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 16, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b254d61d4f', 'id': 'chatcmpl-DIwfGZ8aT6XzEFCGvfIe7Pl4WRYVH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce757-4125-7142-92f1-aae8accee161-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 101, 'total_tokens': 117, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, '

In [13]:
# 방법 2 pickle로 dict를 불러와서 비동기 실행을 한다.
dumpd_chain = dumpd(chain)
with open("fruit_chain_dumpd.pkl", 'wb') as f:
    pickle.dump(dumpd_chain, f)

with open("fruit_chain_dumpd.pkl", 'rb') as f:
    loaded_chain=pickle.load(f)

chain_from_file = load(loaded_chain, allowed_objects='all')
await chain_from_file.ainvoke({"fruit": "사과"})

C:\Users\minem\AppData\Local\Temp\ipykernel_4332\2838784415.py:9: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  chain_from_file = load(loaded_chain, allowed_objects='all')


AIMessage(content='사과의 색상은 종류에 따라 다양합니다. 일반적으로 빨간색, 녹색, 노란색, 혹은 이들이 섞인 색상으로 나타납니다. 예를 들어, 레드 딜리셔스 사과는 진한 빨간색을 띠고 있으며, 그랜니 스미스 사과는 밝은 녹색입니다. 골든 딜리셔스 사과는 노란색을 띠고 있습니다. 어떤 사과는 두 가지 이상의 색상을 가질 수도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 16, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c55974138f', 'id': 'chatcmpl-DIwgcKrvKW8OK1l8JMsuyWBq4eLGI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce758-8a92-7063-ab47-f80cae575209-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 114, 'total_tokens': 130, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'out

In [7]:
# 허깅페이스 로컬
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="EleutherAI/gpt-neo-125M",
    task='text-generation',
    pipeline_kwargs={
        'max_new_tokens': 50,
        'top_k': 50,
        'temperature': 0.1
    }
)

answer = llm.invoke('HuggingFace is')
print(answer)

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HuggingFace is a free-to-use site that uses a strict mode of operation for sharing copyrighted content. We do not necessarily represent or warrant that anyone who tries to use this page will be held accountable for its content. We do not necessarily represent or warrant that


In [8]:
answer = llm.invoke("HuggingFace is")
print(answer)

HuggingFace is a free-to-use site that uses a strict mode of operation for sharing copyrighted content. We do not necessarily represent or warrant that anyone who tries to use this page will be held accountable for its content. We do not necessarily represent or warrant that


In [10]:
from langchain_core.prompts import PromptTemplate

template = """
TEXT에 주어질 장문을 중요한 요점을 뽑아서 요약하시오.

TEXT:
{text}
"""

prompt = PromptTemplate.from_template(template)
chain = prompt | llm

text = """A Large Language Model (LLM) like me, ChatGPT, is a type of artificial intelligence (AI) model designed to understand, generate, and interact with human language. These models are "large" because they're built from vast amounts of text data and have billions or even trillions of parameters. Parameters are the aspects of the model that are learned from training data; they are essentially the internal settings that determine how the model interprets and generates language. LLMs work by predicting the next word in a sequence given the words that precede it, which allows them to generate coherent and contextually relevant text based on a given prompt. This capability can be applied in a variety of ways, from answering questions and composing emails to writing essays and even creating computer code. The training process for these models involves exposing them to a diverse array of text sources, such as books, articles, and websites, allowing them to learn language patterns, grammar, facts about the world, and even styles of writing. However, it's important to note that while LLMs can provide information that seems knowledgeable, their responses are generated based on patterns in the data they were trained on and not from a sentient understanding or awareness. The development and deployment of LLMs raise important considerations regarding accuracy, bias, ethical use, and the potential impact on various aspects of society, including employment, privacy, and misinformation. Researchers and developers continue to work on ways to address these challenges while improving the models' capabilities and applications."""

answer = chain.invoke({'text':text})


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
print(answer)


TEXT에 주어질 장문을 중요한 요점을 뽑아서 요약하시오.

TEXT:
A Large Language Model (LLM) like me, ChatGPT, is a type of artificial intelligence (AI) model designed to understand, generate, and interact with human language. These models are "large" because they're built from vast amounts of text data and have billions or even trillions of parameters. Parameters are the aspects of the model that are learned from training data; they are essentially the internal settings that determine how the model interprets and generates language. LLMs work by predicting the next word in a sequence given the words that precede it, which allows them to generate coherent and contextually relevant text based on a given prompt. This capability can be applied in a variety of ways, from answering questions and composing emails to writing essays and even creating computer code. The training process for these models involves exposing them to a diverse array of text sources, such as books, articles, and websites, allowing them to 

In [13]:
# transformer에서 바로 사용가능
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = 'EleutherAI/gpt-neo-125M'
tokenizer = AutoTokenizer.from_pretrained(
    model_id
)
model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline("text-generation", model=model,
tokenizer=tokenizer, max_new_tokens=512)

hf = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
# chain = prompt | hf | StrOutputParser() 이렇게 사용가능